In [ ]:
# Mise en place SQL
import duckdb
%load_ext sql

In [ ]:
%config SqlMagic.displaylimit = 100

In [ ]:
%sql duckdb:///../data/istex-search-metadata.db

In [ ]:
import matplotlib as mpl
import locale

# Set the locale to French
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
mpl.rcParams['axes.formatter.use_locale'] = True

global cmap 
cmap = mpl.colormaps['viridis']

# Building dir structure

import os
if not os.path.exists('Figures'):
    os.mkdir('Figures')


# Echantillonnage

## Records

In [ ]:
%%sql
SELECT
  *
FROM
  records USING SAMPLE 1000 (reservoir);

## Metadata

In [ ]:
%%sql
SELECT
  *
FROM
  metadata USING SAMPLE 1000 (reservoir);

## Typologie des fichiers produits et traités par la chaine d'ingestion

In [ ]:
%%sql

WITH cte_extension_fulltext AS (
    SELECT 'fulltext' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM fulltext
),
cte_extension_metadata AS (
    SELECT 'metadata' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM metadata
),
cte_extension AS (
    SELECT * FROM cte_extension_fulltext
    UNION ALL
    SELECT * FROM cte_extension_metadata
)
    
SELECT extension, type, COUNT(*) AS "Nombre de fichiers" FROM cte_extension
GROUP BY extension, type;

---
# Analyse

## Documents créés par la chaine d'ingestion Istex

In [ ]:
%%sql
SELECT
  COUNT(*) AS "Documents créés par la chaine d'ingestion Istex"
FROM
  (
    SELECT
      "path"
    FROM
      metadata
    WHERE
      original = FALSE
    UNION ALL
    SELECT
      "path"
    FROM
      fulltext
    WHERE
      original = FALSE
  );

## Analyse des fichiers OCR natifs

## Analyse des fichiers OCR produits

In [ ]:
%%sql
-- Liste des fichiers OCR
SELECT
  *
FROM
  fulltext
  JOIN records USING (arkIstex)
WHERE
  PATH LIKE '%.ocr';

In [ ]:
%%sql
-- Liste des fichiers OCR répartis par décennie de publication du document
SELECT
  decade (try_strptime (publicationDate, '%Y')) * 10 AS decade,
  COUNT(DISTINCT r.arkIstex) AS nb_total,
  COUNT(
    DISTINCT CASE
      WHEN f.path LIKE '%.ocr' THEN f.arkIstex
    END
  ) AS nb_ocr
FROM
  records r
  LEFT JOIN fulltext f USING (arkIstex)
WHERE
  try_strptime (publicationDate, '%Y') IS NOT NULL
GROUP BY
  decade
HAVING
  decade > 1455
  AND decade < 2026
ORDER BY
  decade;

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter
import matplotlib as mpl
import numpy as np
import locale
import os

# Set the locale to French
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
mpl.rcParams['axes.formatter.use_locale'] = True

df = _.DataFrame()


line_color = cmap(0.2)
bar_color  = cmap(0.7)

# Create figure
fig, ax1 = plt.subplots(figsize=(12, 6))

# Left axis — total documents (line)
ax1.plot(df['decade'], df['nb_total'],
         color=line_color, marker='o', linewidth=2, zorder=3,
         label='Nombre total de documents')
ax1.set_xlabel('Décennie de publication')
ax1.tick_params(axis='y', labelcolor=line_color)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Use ScalarFormatter to avoid scientific notation and respect locale
ax1.yaxis.set_major_formatter(
    ScalarFormatter(useOffset=False, useMathText=False)
)
ax1.ticklabel_format(style='plain', axis='y')  # Ensure no scientific notation

# Right axis — OCR documents (bars)
ax2 = ax1.twinx()
ax2.bar(df['decade'], df['nb_ocr'],
        width=5, color=bar_color, alpha=0.8, zorder=2,
        label='Documents OCR « rétrospectifs »')
ax2.tick_params(axis='y', labelcolor=bar_color)

# Use ScalarFormatter for the right axis as well
ax2.yaxis.set_major_formatter(
    ScalarFormatter(useOffset=False, useMathText=False)
)
ax2.ticklabel_format(style='plain', axis='y')  # Ensure no scientific notation

# Ensure line is above bars
ax1.set_zorder(2)
ax2.set_zorder(1)
ax1.patch.set_alpha(0)

# Calcul des totaux
total_docs = int(df['nb_total'].sum())
total_ocr  = int(df['nb_ocr'].sum())

# Combined legend avec totaux formatés en locale française
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

labels1 = [f"{labels1[0]} — {total_docs:n}"]
labels2 = [f"{labels2[0]} — {total_ocr:n}"]

ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# X-axis formatting
ax1.set_xticks(df['decade'])
ax1.set_xticklabels(df['decade'], rotation=45, ha='right')
ax1.set_xlim(left=1800, right=2030)

# Title
#plt.title('Évolution du nombre de documents océrisés rétrospectivements dans ISTEX (par décennies de publication)')

ax1.set_title(
    'Évolution du nombre de documents océrisés rétrospectivement dans ISTEX (par décennies de publication)',
    fontweight="bold", fontsize=11,
)

# Layout + save
plt.tight_layout()
fig.savefig('./Figures/evol_ocr_istex.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
%%sql
SELECT
  LOWER(list_last (split (path, '.'))) AS extension,
  COUNT(*) AS "count"
FROM
  fulltext
GROUP BY
  extension,
  mime
HAVING
  mime = 'text/plain'
ORDER BY
  "count" DESC;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

df = _.DataFrame()

colors = [cmap(i) for i in np.linspace(0.2, 0.7, len(df))]

# Figure
plt.figure(figsize=(12, 10))

# Donut chart
wedges, texts = plt.pie(
    df['count'],
    labels=None,
    startangle=90,
    colors=colors,
    wedgeprops={'edgecolor': 'w', 'linewidth': 1, 'width': 0.4}
)

# Custom labels outside with percentages and counts
total = df['count'].sum()
for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1) / 2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))
    horizontalalignment = 'left' if x > 0 else 'right'
    connectionstyle = f"angle,angleA=0,angleB={ang}"

    plt.annotate(
        f"{df['extension'].iloc[i]} : {locale.format_string('%d', df['count'].iloc[i], grouping=True)} "
        f"({locale.format_string('%.1f', df['count'].iloc[i] / total * 100, grouping=True)}\u202f%)",
        xy=(x * 0.7, y * 0.7),
        xytext=(1.2 * x, 1.2 * y),
        horizontalalignment=horizontalalignment,
        fontsize=12,
        weight='bold',
        arrowprops=dict(arrowstyle='-', color='gray', connectionstyle=connectionstyle)
    )

# Center total
plt.text(
    0, 0,
    f"Total\n{locale.format_string('%d', total, grouping=True)}",
    ha='center',
    va='center',
    fontsize=16,
    weight='bold'
)

# Title
plt.title('Répartition des fichiers par extension dans ISTEX', fontsize=18, pad=20, weight='bold')
plt.tight_layout()
plt.savefig('./Figures/donut_extensions_improved.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
%sql duckdb:///../data/output.db

In [ ]:
%%sql

SET VARIABLE threshold = 0.8;

SELECT
    decade                                                                                       AS decade,
    COUNT(*)                                                                                     AS total,
    COUNT(*) FILTER (WHERE avg_text_intersecting_image_pct >  getvariable('threshold'))          AS nb_ocr,
    (array_agg(arkIstex) FILTER (WHERE avg_text_intersecting_image_pct >  getvariable('threshold')))[:10] AS ocr_sample,
    COUNT(*) FILTER (WHERE avg_text_intersecting_image_pct <= getvariable('threshold'))          AS nb_non_ocr,
    (array_agg(arkIstex) FILTER (WHERE avg_text_intersecting_image_pct <= getvariable('threshold')))[:10] AS non_ocr_sample
FROM output."data"
WHERE pdfPageCount > 3
  AND pdfWordCount > 500
GROUP BY decade
ORDER BY decade;

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import matplotlib as mpl
import numpy as np
import locale

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
mpl.rcParams['axes.formatter.use_locale'] = True

df = _.DataFrame()

THRESHOLD = 0.8
MIN_WORDS = 500
MIN_PAGES = 3
PDF_YEAR = 1993

decades = df["decade"].values
nb_ocr  = df["nb_ocr"].values
total   = df["total"].values

total_docs = int(total.sum())
ocr_docs   = int(nb_ocr.sum())

ocr_color = cmap(0.7)  # Color for OCR documents
total_color = cmap(0.2)  # Color for total documents
pdf_line_color = cmap(0.0)  # Color for the PDF vertical line

# --- First Figure ---
fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(decades, nb_ocr, alpha=0.3, color=ocr_color)
ax.plot(decades, nb_ocr, color=ocr_color, lw=2,
        label=f"Documents OCR « natifs » (seuil {THRESHOLD}) — {ocr_docs:n}")
ax.plot(decades, total, color=total_color, lw=1.5, ls="--",
        label=f"Total documents — {total_docs:n}")

ax.axvline(PDF_YEAR, color=pdf_line_color, linestyle=":", lw=1.5,
           label="Diffusion publique du format PDF (1993)")

ax.set_xlabel("Décennies de publication")
ax.set_ylabel("Nombre total de documents")
ax.set_title(
    f"Nombre de documents OCR « natifs » par décennie (seuil = {THRESHOLD})\n"
    f"Documents avec > {MIN_WORDS} mots et > {MIN_PAGES} pages",
    fontweight="bold", fontsize=11,
)

ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False, useMathText=False))
ax.ticklabel_format(style='plain', axis='y')

ax.set_xticks(decades)
ax.set_xticklabels([str(d) for d in decades], rotation=45, ha="right", fontsize=8)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)

out_path = "./Figures/ocr_area.png"
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)

# --- Second Figure ---
START_YEAR = 1980
mask   = decades >= START_YEAR
dec2   = decades[mask]
ocr2   = nb_ocr[mask]
total2 = total[mask]

total_docs2 = int(total2.sum())
ocr_docs2   = int(ocr2.sum())

fig2, ax2 = plt.subplots(figsize=(13, 5))

ax2.fill_between(dec2, ocr2, alpha=0.3, color=ocr_color)
ax2.plot(dec2, ocr2, color=ocr_color, lw=2, marker="o", ms=5,
         label=f"Documents OCR « natifs » (seuil {THRESHOLD}) — {ocr_docs2:n}")
ax2.plot(dec2, total2, color=total_color, lw=1.5, ls="--", marker="o", ms=5,
         label=f"Total documents — {total_docs2:n}")

ax2.axvline(PDF_YEAR, color=pdf_line_color, linestyle=":", lw=1.5,
            label="Diffusion publique du format PDF (1993)")

ax2.set_xlabel("Décennies de publication")
ax2.set_ylabel("Nombre total de documents")
ax2.set_title(
    f"Nombre de documents OCR « natifs » — {START_YEAR} à aujourd'hui (seuil = {THRESHOLD})\n"
    f"Documents avec > {MIN_WORDS} mots et > {MIN_PAGES} pages",
    fontweight="bold", fontsize=11,
)

ax2.yaxis.set_major_formatter(ScalarFormatter(useOffset=False, useMathText=False))
ax2.ticklabel_format(style='plain', axis='y')

ax2.set_xticks(dec2)
ax2.set_xticklabels([str(d) for d in dec2], rotation=45, ha="right", fontsize=9)
ax2.legend()
ax2.spines[["top", "right"]].set_visible(False)

out_path2 = f"./Figures/ocr_area_{START_YEAR}.png"
fig2.savefig(out_path2, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig2)